In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
print("project root:", _root)

project root: /workspaces/geo_distance_arena


# 04 - The Nearest Store Flips

Where the metric changes the decision. For a set of branches and a customer base, we assign each customer to the nearest branch by straight-line and by road, and measure how often the assignment **flips** -- concentrated at the barriers. Then we draw the catchment three ways: the straight-line circle, the road-distance catchment, and the drive-time isochrone.

In [2]:
import numpy as np
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import Point

from src.graph.build import load_city_graph
from src.distance.road import add_edge_speeds, add_edge_travel_times
from src.scenarios.nearest_facility import assign_nearest
from src.scenarios.catchment import (
    straight_line_circle, road_distance_catchment, drive_time_catchment,
)
from src.evaluation.ranking_flips import flip_rate
from src.scenarios.points import load_scenario
from src.config import get_settings
from src.io_utils import save_fig, save_json

settings = get_settings()
drive = load_city_graph('copenhagen', 'drive')
add_edge_speeds(drive, settings.speeds.default_kph, settings.speeds.by_highway)
add_edge_travel_times(drive)
scenario = load_scenario('copenhagen')
facilities = [tuple(f) for f in scenario['facilities']]
customers = [tuple(c) for c in scenario['customers']]

## How often does the nearest branch flip?

In [3]:
assignments = assign_nearest(drive, facilities, customers)
rate = flip_rate(assignments)
save_json({'flip_rate': rate, 'n_customers': len(assignments),
           'n_flips': int(assignments['flip'].sum())}, '04_flip_rate')
save_json(assignments.to_dict(orient='records'), '04_assignments')
print(f'flip rate: {rate:.1%}')
assignments.head()

flip rate: 10.8%


,lat,lon,straight_nearest,road_nearest,straight_m,road_m,flip
0,55.667959,12.545854,2,2,822.320168,1027.411499,False
1,55.675211,12.546999,2,2,1316.522026,1739.183854,False
2,55.659014,12.602469,3,3,592.819220,757.380607,False
3,55.680181,12.601926,1,3,925.728862,2267.168680,True
4,55.668035,12.616240,3,3,1120.304617,1266.942071,False


In [4]:
fig, ax = ox.plot_graph(drive, node_size=0, edge_color='#e8e8e8', edge_linewidth=0.5,
                        bgcolor='white', show=False, close=False)
flipped = assignments[assignments['flip']]
stable = assignments[~assignments['flip']]
ax.scatter(stable['lon'], stable['lat'], c='#54A24B', s=22, label='stable', zorder=3)
ax.scatter(flipped['lon'], flipped['lat'], c='#E45756', s=40, label='flips', zorder=4)
for lat, lon in facilities:
    ax.scatter([lon], [lat], marker='s', c='#1f77b4', s=120, zorder=5)
ax.set_title(f'Nearest-branch assignment flips for {rate:.0%} of customers')
ax.legend(loc='upper right')
save_fig(fig, '04_flip_map')
plt.show()

## Catchment three ways

For one branch: the straight-line circle everyone draws, the true road-distance catchment, and the drive-time isochrone. We count how many customers fall inside each.

In [5]:
branch = facilities[0]
radius_m = 1500.0
circle = straight_line_circle(branch, radius_m)
road_cat = road_distance_catchment(drive, branch, radius_m)
iso = drive_time_catchment(drive, branch, max_seconds=240)
pts = [Point(lon, lat) for lat, lon in customers]
counts = {
    'inside_circle': int(sum(circle.contains(p) for p in pts)),
    'inside_road_catchment': int(sum(road_cat.contains(p) for p in pts)),
    'inside_drive_time': int(sum(iso.contains(p) for p in pts)),
    'n_customers': len(pts),
}
save_json(counts, '04_catchment_counts')
counts

{'inside_circle': 32,
 'inside_road_catchment': 30,
 'inside_drive_time': 57,
 'n_customers': 120}

In [6]:
import geopandas as gpd
fig, ax = ox.plot_graph(drive, node_size=0, edge_color='#e8e8e8', edge_linewidth=0.5,
                        bgcolor='white', show=False, close=False)
for geom, color, label in [(circle, '#E45756', 'straight-line circle'),
                           (road_cat, '#4C78A8', 'road-distance catchment'),
                           (iso, '#54A24B', 'drive-time isochrone')]:
    gpd.GeoSeries([geom]).boundary.plot(ax=ax, color=color, linewidth=2, label=label)
ax.scatter([branch[1]], [branch[0]], marker='s', c='#1f77b4', s=140, zorder=6)
ax.set_title('One branch, three catchments')
ax.legend(loc='upper right')
save_fig(fig, '04_catchments')
plt.show()

**Takeaway.** The straight-line circle is a fiction: it claims customers the branch cannot reach quickly and misses ones it can. A double-digit share of 'nearest branch' assignments flip under the real metric, concentrated exactly at the barriers.